# 🤖 Smart CV Analyzer: Tahap 3 - Rule-Based Auto BIO Tagging

Setelah kita memiliki "kanvas kosong" berupa file `.jsonl` dengan label `"O"` di Tahap 2, sekarang kita akan mengisi label tersebut secara otomatis menggunakan **Rule-Based Tagging**.

Langkah-langkah di notebook ini:
1. **Knowledge Base:** Menentukan kamus kata untuk SKILL, ROLE, EDU, CERT, COMP, dan EXP.
2. **BIO Logic:** Membuat fungsi yang bisa membedakan kapan harus memberi tag `B-` (awal kata) dan `I-` (lanjutan kata).
3. **Auto-Tagging:** Memproses file `cv_dataset_ready_to_tag.jsonl` dan menyimpannya menjadi dataset yang sudah terlabeli.

## 1. Import Library
Kita hanya membutuhkan `json` untuk memanipulasi file `.jsonl` dan `re` (RegEx) jika diperlukan untuk pencocokan pola lanjutan.

In [1]:
import json
import re

## 2. Definisi Kamus Kata (Knowledge Base)
Mendefinisikan kumpulan kata kunci untuk setiap label entitas.

In [2]:
# Definisi Kamus Kata
dict_skills = {"python", "sql", "docker", "dbt", "apache", "spark", "etl", "bigquery", "gcp", "kubernetes", "redshift", "pytorch", "tensorflow", "numpy", "scikit-learn", "nlp", "machine", "learning", "tableau", "power", "bi", "excel"}
dict_roles = {"data", "engineer", "scientist", "analyst", "developer", "associate", "manager", "lead", "intern"}
dict_edus = {"s1", "s2", "sarjana", "informatika", "sistem", "informasi", "teknik", "universitas", "institut", "indonesia", "ipk"}
dict_comps = {"xendit", "kredivo", "tokopedia", "gojek", "grab", "shopee", "bca", "mandiri", "telkomsel", "kudo"}
dict_certs = {"dicoding", "coursera", "udemy", "sertifikasi", "bootcamp", "hackathon", "google", "cloud", "certified"}

# Fungsi khusus untuk mendeteksi durasi atau tanggal (EXP)
def is_exp_or_date(word):
    word = word.lower()
    # Deteksi tahun (2019, 2023) atau angka durasi singkat (1-5 tahun)
    if word.isdigit() and (len(word) == 4 or (int(word) > 0 and int(word) < 11)):
        return True
    # Deteksi bulan atau penunjuk waktu
    if word in ["tahun", "bulan", "januari", "februari", "maret", "april", "mei", "juni", "juli", "agustus", "september", "oktober", "november", "desember"]:
        return True
    return False

## 3. Fungsi Logika BIO Tagging
Fungsi ini akan memeriksa setiap kata (token). Jika kata tersebut ada di kamus, maka akan diberi tag. Jika ada dua kata berurutan dengan kategori yang sama (misal: "Data" dan "Scientist"), maka kata kedua akan otomatis diberi awalan `I-`.

In [3]:
def auto_bio_tagger(tokens):
    tags = ["O"] * len(tokens)
    prev_entity = "O"
    
    for i, token in enumerate(tokens):
        word = token.lower()
        current_entity = "O"
        
        # Cek kecocokan dengan kamus
        if word in dict_skills: current_entity = "SKILL"
        elif word in dict_roles: current_entity = "ROLE"
        elif word in dict_edus: current_entity = "EDU"
        elif word in dict_comps: current_entity = "COMP"
        elif word in dict_certs: current_entity = "CERT"
        elif is_exp_or_date(word): current_entity = "EXP"
            
        # Aturan BIO (Begin & Inside)
        if current_entity != "O":
            if current_entity == prev_entity:
                tags[i] = f"I-{current_entity}"
            else:
                tags[i] = f"B-{current_entity}"
        
        prev_entity = current_entity
        
    return tags

## 4. Eksekusi Auto-Tagging
Memuat file dari Tahap 2, menjalankan fungsi *tagger*, dan menyimpannya ke file baru bernama `cv_dataset_auto_tagged.jsonl`.

In [4]:
input_filename = "cv_dataset_ready_to_tag.jsonl"
output_filename = "cv_dataset_auto_tagged.jsonl"

final_data = []

try:
    with open(input_filename, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            # Jalankan auto tagger untuk mengganti "O" dengan label asli
            data["ner_tags"] = auto_bio_tagger(data["tokens"])
            final_data.append(data)

    # Simpan ke file baru
    with open(output_filename, "w", encoding="utf-8") as f:
        for entry in final_data:
            # 🌟 REVISI: Tambahkan parameter ensure_ascii=False agar karakter Unicode terbaca normal
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
            
    print(f"✅ Berhasil! Dataset terlabeli disimpan di: {output_filename}")
    print(f"📊 Total kalimat diproses: {len(final_data)}")

except FileNotFoundError:
    print("❌ Error: File 'cv_dataset_ready_to_tag.jsonl' tidak ditemukan. Pastikan sudah menjalankan notebook Tahap 2.")

✅ Berhasil! Dataset terlabeli disimpan di: cv_dataset_auto_tagged.jsonl
📊 Total kalimat diproses: 980


## 5. Verifikasi Hasil
Melihat beberapa contoh hasil pelabelan otomatis untuk memastikan tag `B-` dan `I-` sudah muncul dengan benar dan tidak sekadar `"O"` semua.

In [5]:
# Intip 3 data pertama
for i in range(min(3, len(final_data))):
    print(f"\nKalimat ke-{i+1}:")
    print(f"Tokens: {final_data[i]['tokens']}")
    print(f"Tags  : {final_data[i]['ner_tags']}")


Kalimat ke-1:
Tokens: ['Data', 'engineer', 'yang', 'berpengalaman', 'dalam', 'membangun', 'dan', 'mengelola', 'pipeline', 'data', 'skala', 'besar']
Tags  : ['B-ROLE', 'I-ROLE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ROLE', 'O', 'O']

Kalimat ke-2:
Tokens: ['bekerja', 'dengan', 'berbagai', 'teknologi', 'big', 'data', 'untuk', 'memproses', 'data', 'streaming', 'dan', 'batch']
Tags  : ['O', 'O', 'O', 'O', 'O', 'B-ROLE', 'O', 'O', 'B-ROLE', 'O', 'O', 'O']

Kalimat ke-3:
Tokens: ['dalam', 'merancang', 'arsitektur', 'data', 'warehouse', 'yang', 'efisien']
Tags  : ['O', 'O', 'O', 'B-ROLE', 'O', 'O', 'O']


In [6]:
# Tipe Cell: Code
total_sentences = len(final_data)
all_o_sentences = 0

for row in final_data:
    # Cek apakah semua tag di kalimat ini adalah "O"
    if all(tag == "O" for tag in row["ner_tags"]):
        all_o_sentences += 1

has_entity = total_sentences - all_o_sentences

print("📊 Laporan Keseimbangan Dataset:")
print(f"Total kalimat          : {total_sentences}")
print(f"Kalimat Punya Entitas  : {has_entity} ({(has_entity/total_sentences)*100:.1f}%)")
print(f"Kalimat 'O' Semua      : {all_o_sentences} ({(all_o_sentences/total_sentences)*100:.1f}%)")

📊 Laporan Keseimbangan Dataset:
Total kalimat          : 980
Kalimat Punya Entitas  : 635 (64.8%)
Kalimat 'O' Semua      : 345 (35.2%)
